# 06 — Data integrity and leakage audit

**Purpose.** Check that every predictor row is exactly one calendar day before its target, report
coverage by station and variable, quantify where the submitted pipeline's bidirectional interpolation
touched the data, and export the feature-availability table.

**Answers:** Reviewer 3 comments 1–2 and the minor comment on the final predictor count; Reviewer 4
comments 3, 4 and the missing-data part of comment 10.

**Inputs:** `data/Meteorology_Sarkoy_2014_2024.json`, `data/wind_speed_trakya.csv`,
`data/Meteorology_Ganos_2016_Nov_2024.csv`, `df_final_features.feather`.

**Outputs:** `revision/tables/S3_*.csv`, `S5_feature_availability_*.csv`, `audit_interpolated_targets.csv`,
and values in `revision/revision_results.json`. **Runtime:** ~2 min.

In [1]:
# --- Setup: Drive, paths, shared helpers -------------------------------------------------------
import os, sys, json, time, warnings, subprocess
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'shap', 'scienceplots',
                    'ephem', 'catboost', 'lightgbm'], check=False)
except Exception:
    pass
_BASE = os.environ.get('SARKOY_BASE', '/content/drive/MyDrive/windforecast_rev1/')
sys.path.insert(0, os.path.join(_BASE, 'revision'))
if not os.path.exists(os.path.join(_BASE, 'revision', 'revision_utils.py')):
    raise FileNotFoundError('Copy revision_utils.py into ' + os.path.join(_BASE, 'revision') + ' first.')
import revision_utils as ru
import numpy as np, pandas as pd, matplotlib.pyplot as plt
BASE, REV, IN_COLAB, FAST = ru.get_paths()
ru.set_style()
RES = ru.Results(REV)
RES.path = REV + 'results_06.json'
GPU = ru.has_gpu()
try:
    from IPython.display import display
except Exception:
    display = print
print('BASE:', BASE, '| REV:', REV, '| GPU:', GPU, '| FAST (smoke test):', FAST)
print(ru.log_versions(REV))


Mounted at /content/drive
BASE: /content/drive/MyDrive/windforecast_rev1/ | REV: /content/drive/MyDrive/windforecast_rev1/revision/ | GPU: False | FAST (smoke test): False
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'catboost': '1.2.10', 'optuna': '5.0.0', 'shap': '0.52.0', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0', 'seaborn': '0.13.2', 'pyarrow': '23.0.1', 'ephem': '4.2.1', 'tensorflow': '2.20.0', 'gpu': False}


In [2]:
# --- Raw sources: date coverage ----------------------------------------------------------------
num, dropped10, sar, trk, gan = ru.raw_merged(BASE)
rows = []
for name, d in [('Şarköy', sar), ('Trakya network', trk), ('Ganos', gan)]:
    cal = pd.date_range(d.index.min(), d.index.max(), freq='D')
    rows.append(dict(source=name, start=str(d.index.min().date()), end=str(d.index.max().date()),
                     rows=len(d), calendar_days=len(cal),
                     missing_dates=int(len(cal.difference(d.index))),
                     duplicate_dates=int(d.index.duplicated().sum())))
src_cov = pd.DataFrame(rows)
display(src_cov)
cal = pd.date_range(num.index.min(), num.index.max(), freq='D')
print('merged record:', num.index.min().date(), '->', num.index.max().date(), '|', len(num), 'rows |',
      len(cal), 'calendar days |', len(cal.difference(num.index)), 'missing dates')
print('columns dropped for >10% gaps:', len(dropped10))
src_cov.to_csv(REV + 'tables/S3_source_coverage.csv', index=False)


,source,start,end,rows,calendar_days,missing_dates,duplicate_dates
0,Şarköy,2014-01-01,2024-12-31,4018,4018,0,0
1,Trakya network,2015-01-01,2024-12-31,3653,3653,0,0
2,Ganos,2016-11-01,2024-12-31,2983,2983,0,0


merged record: 2016-11-01 -> 2024-12-31 | 2983 rows | 2983 calendar days | 0 missing dates
columns dropped for >10% gaps: 24


In [3]:
# --- Coverage by station and variable inside the study window ----------------------------------
win = pd.date_range(num.index.min(), num.index.max(), freq='D')
rows = []
for c in sar.columns:
    rows.append(('Şarköy', c, float(sar[c].reindex(win).isna().mean() * 100), ru.max_gap(sar[c].reindex(win).isna())))
for c in gan.columns:
    rows.append(('Ganos', c, float(gan[c].reindex(win).isna().mean() * 100), ru.max_gap(gan[c].reindex(win).isna())))
for s in ru.STATIONS_ALL:
    for v in ('mean_speed', 'max_speed', 'mean_wind_angle', 'max_wind_angle'):
        c = f'{v}_{s}'
        if c in trk.columns:
            col = pd.to_numeric(trk[c], errors='coerce').reindex(win)
            rows.append((ru.STATION_LABEL[s], c, float(col.isna().mean() * 100), ru.max_gap(col.isna())))
cov = pd.DataFrame(rows, columns=['station', 'variable', 'missing_pct', 'longest_gap_days'])
cov.to_csv(REV + 'tables/S3_coverage_by_variable.csv', index=False)
st = cov.groupby('station')['missing_pct'].agg(['mean', 'max']).round(2)
st['longest_gap_days'] = cov.groupby('station')['longest_gap_days'].max()
st['used_in_models'] = ['no (>10% missing)' if i in [ru.STATION_LABEL[x] for x in ru.STATIONS_DROPPED]
                        else 'yes' for i in st.index]
st.to_csv(REV + 'tables/S3_coverage_by_station.csv')
display(st)
RES.put('N_STATIONS_USED', len(ru.STATIONS_USED)).put('N_STATIONS_DROPPED', len(ru.STATIONS_DROPPED))
RES.put('COV_MISSING_MAX_USED', st.loc[[i for i in st.index if st.loc[i, 'used_in_models'] == 'yes'], 'max'].max(), '.1f')


,mean,max,longest_gap_days,used_in_models
station,,,,
Edirne,0.17,0.67,15,yes
Enez,0.44,0.47,6,yes
Ganos,0.93,1.07,9,yes
Hamzabeyli,15.97,15.99,426,no (>10% missing)
Havalimanı,22.90,32.62,742,no (>10% missing)
Havsa,1.63,1.68,10,yes
Hayrabolu,0.28,0.30,4,yes
Kemal,15.69,15.69,426,no (>10% missing)
Keşan,2.65,2.68,51,yes


In [4]:
# --- Calendar audit of the modelling matrix and the one-day target offset ----------------------
pub = ru.load_matrix(BASE, REV, 'published')
idx = pub.index
steps = pd.Series((idx[1:] - idx[:-1]).days)
nxt = idx + pd.Timedelta(days=1)
raw_next = sar[ru.SOURCE].reindex(nxt).to_numpy()
tgt = pub[ru.TARGET].to_numpy()
observed = ~np.isnan(raw_next)
mismatch = int((~np.isclose(tgt[observed], raw_next[observed], atol=1e-6)).sum())
internal = bool(np.allclose(pub[ru.TARGET].to_numpy()[:-1], pub[ru.SOURCE].to_numpy()[1:], atol=1e-6))
split = np.where(idx.year < 2023, 'train', np.where(idx.year == 2023, 'validation', 'test'))
aud = pd.DataFrame({'date': idx, 'split': split, 'target_date': nxt, 'target_observed': observed})
summary = aud.groupby('split').agg(first=('date', 'min'), last=('date', 'max'), days=('date', 'size'),
                                   interpolated_targets=('target_observed', lambda s: int((~s).sum())))
display(summary)
print('all consecutive steps = 1 day:', bool((steps == 1).all()), '| unique dates:', idx.is_unique,
      '| target equals next-day record on observed days: ', mismatch == 0,
      '| internal consistency target(t)=WS(t+1):', internal)
aud[~aud.target_observed].to_csv(REV + 'tables/audit_interpolated_targets.csv', index=False)
summary.to_csv(REV + 'tables/S3_calendar_audit.csv')
RES.put('AUDIT_START', idx.min().strftime('%d %B %Y').lstrip('0')).put('AUDIT_END', idx.max().strftime('%d %B %Y').lstrip('0'))
RES.put('AUDIT_N', f'{len(idx):,}').put('AUDIT_STEP_OK', 'yes' if (steps == 1).all() else 'NO')
RES.put('AUDIT_N_TRAIN', int((split == 'train').sum()), 'd').put('AUDIT_N_VAL', int((split == 'validation').sum()), 'd')
RES.put('AUDIT_N_TEST', int((split == 'test').sum()), 'd').put('AUDIT_TGT_MISMATCH', mismatch, 'd')
RES.put('TGT_INTERP_N', int((~observed).sum()), 'd')
RES.put('TGT_INTERP_TEST', int(summary.loc['test', 'interpolated_targets']) if 'test' in summary.index else 0, 'd')
RES.put('AUDIT_YEARS', round((idx.max() - idx.min()).days / 365.25, 1), '.1f')


Loaded published matrix: (2964, 564), 2016-11-19 -> 2024-12-30, observed targets: 2964


,first,last,days,interpolated_targets
split,,,,
test,2024-01-01,2024-12-30,365,1
train,2016-11-19,2022-12-31,2234,5
validation,2023-01-01,2023-12-31,365,1


all consecutive steps = 1 day: True | unique dates: True | target equals next-day record on observed days:  True | internal consistency target(t)=WS(t+1): True


In [5]:
# --- How much did the submitted (bidirectional) interpolation touch? ---------------------------
M = num.isna()
rowflag = M.any(axis=1)
yr = M.index.year
splits = {'train (<2023)': yr < 2023, 'validation (2023)': yr == 2023, 'test (2024)': yr == 2024}
rows = []
for k, m in splits.items():
    rows.append(dict(split=k, days=int(m.sum()), days_with_any_gap=int(rowflag[m].sum()),
                     pct_days_with_gap=round(100 * rowflag[m].mean(), 2),
                     pct_cells_filled=round(100 * M[m].to_numpy().mean(), 3)))
gaps = pd.DataFrame(rows)
display(gaps)
longest = max(ru.max_gap(M[c]) for c in M.columns)
lead = int(sum(M[c].iloc[0] for c in M.columns)); trail = int(sum(M[c].iloc[-1] for c in M.columns))
print('longest gap (days):', longest, '| columns with a gap at the very start:', lead, '| at the very end:', trail)
gaps.to_csv(REV + 'tables/S5_interpolation_exposure.csv', index=False)
flags = ru.missing_flags(BASE)
flags.reset_index().rename(columns={'index': 'date_time'}).to_feather(REV + 'missing_flags.feather')
RES.put('INTERP_CELLS_PCT', 100 * M.to_numpy().mean(), '.2f')
for k, lab in [('train (<2023)', 'TRAIN'), ('validation (2023)', 'VAL'), ('test (2024)', 'TEST')]:
    RES.put('INTERP_ROWS_PCT_' + lab, float(gaps.loc[gaps.split == k, 'pct_days_with_gap'].iloc[0]), '.1f')
RES.put('INTERP_MAX_GAP', longest, 'd').put('N_COLS_DROPPED_10PCT', len(dropped10), 'd')


,split,days,days_with_any_gap,pct_days_with_gap,pct_cells_filled
0,train (<2023),2252,397,17.63,1.074
1,validation (2023),365,126,34.52,1.744
2,test (2024),366,275,75.14,4.581


longest gap (days): 109 | columns with a gap at the very start: 0 | at the very end: 53


In [6]:
# --- Where are the gaps? Runs of missing values, classified as internal / leading / trailing --------
# An internal gap bridged by the submitted linear interpolation used the next valid observation (look-ahead);
# a trailing gap was filled with the last valid value (no look-ahead); a leading gap borrowed the first
# valid value (look-ahead).
runs = []
for c in M.columns:
    s = M[c]
    if not s.any():
        continue
    grp = (s != s.shift()).cumsum()
    for _, blk in s[s].groupby(grp[s]):
        a, b = blk.index.min(), blk.index.max()
        before = num[c].loc[:a - pd.Timedelta(days=1)].last_valid_index()
        after = num[c].loc[b + pd.Timedelta(days=1):].first_valid_index()
        kind = 'internal' if (before is not None and after is not None) else ('leading' if before is None else 'trailing')
        st = ru._station_in(c) or ('ganos' if 'ganos' in c else 'sarkoy')
        runs.append(dict(variable=c, station=ru.STATION_LABEL.get(st, st), start=a.date(), end=b.date(),
                         days=len(blk), kind=kind, test_year_days=int((blk.index.year == 2024).sum())))
GR = pd.DataFrame(runs)
st_runs = (GR.groupby(['station', 'start', 'end', 'days', 'kind', 'test_year_days']).size()
             .rename('variables').reset_index().sort_values('days', ascending=False))
st_runs.to_csv(REV + 'tables/S5_gap_runs.csv', index=False)
display(st_runs.head(12))
y24 = M.index.year == 2024
local_cols = [c for c in M.columns if ru._station_in(c) is None]
cells24 = GR.groupby('kind').test_year_days.sum()
lt = st_runs[st_runs.test_year_days > 0].iloc[0] if (st_runs.test_year_days > 0).any() else None
RES.put('GAP_TEST_LOCAL_DAYS', int(M.loc[y24, local_cols].any(axis=1).sum()), 'd')
RES.put('GAP_TEST_REGIONAL_DAYS', int(M.loc[y24, [c for c in M.columns if c not in local_cols]].any(axis=1).sum()), 'd')
RES.put('GAP_TEST_INTERNAL_PCT', 100 * cells24.get('internal', 0) / max(cells24.sum(), 1), '.0f')
if lt is not None:
    RES.put('GAP_TEST_LONGEST', f"{lt.days} days at {lt.station}, {pd.Timestamp(lt.start).day} {pd.Timestamp(lt.start):%B}–{pd.Timestamp(lt.end).day} {pd.Timestamp(lt.end):%B %Y}")
top_st = (GR[GR.test_year_days > 0].groupby('station').test_year_days.max().sort_values(ascending=False))
_names = list(top_st.index[:4])
RES.put('GAP_TEST_STATIONS', ', '.join(_names[:-1]) + ' and ' + _names[-1] if len(_names) > 1 else ''.join(_names))


,station,start,end,days,kind,test_year_days,variables
142,Çerkezköy,2024-06-24,2024-10-10,109,internal,109,8
167,Çorlu,2024-10-20,2024-12-31,73,trailing,73,4
49,Keşan,2018-12-02,2019-01-21,51,internal,0,8
141,Çerkezköy,2024-04-06,2024-05-15,40,internal,40,8
200,Şarköy,2020-10-02,2020-10-31,30,internal,0,3
136,Çerkezköy,2023-02-14,2023-03-07,22,internal,0,7
120,Uzunköprü,2023-11-07,2023-11-27,21,internal,0,7
121,Uzunköprü,2023-11-08,2023-11-27,20,internal,0,1
100,Saray,2024-05-22,2024-06-10,20,internal,20,8
75,Malkara,2024-09-19,2024-10-06,18,internal,18,8


In [7]:
# --- Feature availability table (Reviewer 3, comment 1) ----------------------------------------
import re
def window_of(f):
    m = re.search(r'roll_(\d+)|_ema_(\d+)|vol_season_(?:sin|cos)_(\d+)', f)
    if m:
        w = next(g for g in m.groups() if g)
        return f'trailing {w} days (ends on day t)'
    if f.startswith('corr_sarkoy_'):
        return 'trailing 7 days (ends on day t)'
    if f.startswith('lag_2_') or f.endswith('_lag_2'):
        return 'day t−2'
    if f.startswith('lag_1_') or f.endswith('_lag_1') or f.startswith('lag1_'):
        return 'day t−1'
    if 'annual_cum' in f:
        return '1 January to day t'
    if 'days_since' in f or 'streak' in f or 'consecutive' in f:
        return 'all days up to day t'
    if ru.feature_group(f) in ('calendar', 'lunar'):
        return 'calendar date of day t+1 (deterministic)'
    return 'day t'
feat = pd.DataFrame({'feature': ru.FEATURES_559})
feat['label'] = [ru.pretty(f) for f in feat.feature]
feat['group'] = [ru.feature_group(f) for f in feat.feature]
feat['historical_window'] = [window_of(f) for f in feat.feature]
feat['known_at_forecast_origin'] = 'yes (00:00 local time after day t closes)'
feat['relative_to_origin'] = np.where(feat.group.isin(['calendar', 'lunar']), 'deterministic, known in advance',
                                      'before or at the origin')
feat['submitted_pipeline_uses_future_obs'] = np.where(
    feat.group.isin(['calendar', 'lunar']), 'no',
    'only through bidirectional interpolation of gaps in its source variable')
feat['revised_pipeline_uses_future_obs'] = 'no (forward fill only)'
feat.to_csv(REV + 'tables/S5_feature_availability_all559.csv', index=False)
fam = (feat.groupby(['group', 'historical_window', 'relative_to_origin'])
           .agg(n=('feature', 'size'), examples=('label', lambda s: '; '.join(list(s)[:3])))
           .reset_index().sort_values('n', ascending=False))
fam.to_csv(REV + 'tables/S5_feature_availability_family.csv', index=False)
display(fam.head(20))
print('predictors:', len(feat), '| families:', fam.group.nunique())
RES.put('N_PREDICTORS', len(feat), 'd').save()


,group,historical_window,relative_to_origin,n,examples
7,local_direction,day t,before or at the origin,70,Sarkoy mean wind dir cat E; Sarkoy mean wind d...
23,local_wind_eng,day t,before or at the origin,43,Max wind angle sin; Max wind angle cos; Mean w...
14,local_met_eng,day t,before or at the origin,22,Sarkoy temp mean to max ratio; Sarkoy temp mea...
0,calendar,calendar date of day t+1 (deterministic),"deterministic, known in advance",19,Day of week; Day of year sin; Day of year cos
3,ganos,day t,before or at the origin,14,Max humidity at ganos; Max temperature at gano...
11,local_met,day t−2,before or at the origin,10,Mean temperature (t−2); Min temperature (t−2);...
10,local_met,day t−1,before or at the origin,10,Mean temperature (t−1); Min temperature (t−1);...
5,ganos,day t−2,before or at the origin,9,Mean temperature at ganos (t−2); Max temperatu...
9,local_met,day t,before or at the origin,9,Max pressure; Max humidity; Max temperature
4,ganos,day t−1,before or at the origin,9,Mean temperature at ganos (t−1); Max temperatu...


predictors: 559 | families: 41
26 values written to /content/drive/MyDrive/windforecast_rev1/revision/results_06.json
